# Phase 2 QRC Temporal Resolution Ablation

The ESN-vs-QRC comparison showed that ESN states are much more target-aligned and much better at high-volatility recall. One likely reason is that ESN processes all 40 daily inputs recurrently, while the current QRC sees only 6 anchors from the 40-day window.

This notebook keeps the best QRC dynamics/readout fixed and varies only temporal resolution:

- anchor_count: 6, 10, 20
- anchor_policy: even, recent

Fixed architecture/readout:

- full TFIM topology
- 3 Trotter steps per anchor
- 3 virtual nodes per anchor
- evolution_time = 0.5
- ZXZZ observables
- disorder_strength = 0.20
- winsorized top-k readout with k = min(120, n_features)
- ridge alpha = 3000

Decision rule: if increasing temporal resolution does not improve prediction variance / high-volatility recall / test metrics, static-window QRC is likely structurally insufficient for this regression task.

In [1]:
from pathlib import Path
import os

if Path.cwd().name == "notebooks":
    os.chdir("..")

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

from qpitome_qrc.data.features import FEATURE_COLUMNS
from qpitome_qrc.data.loaders import load_phase2_volatility_data
from qpitome_qrc.data.pca import fit_transform_pca_splits_train_only
from qpitome_qrc.data.splits import chronological_tabular_split
from qpitome_qrc.evaluation.metrics import evaluate_volatility_forecast
from qpitome_qrc.qrc.tfim_reservoir import (
    TFIMQRCConfig,
    _safe_feature_target_correlations,
    diagnose_reservoir_feature_splits,
    fit_tfim_qrc_regressor,
    make_qrc_sequence_splits,
)

## 1. Data

In [2]:
target = "future_rv_20d"

df = load_phase2_volatility_data()
splits = chronological_tabular_split(df)

pca6 = fit_transform_pca_splits_train_only(
    splits,
    feature_columns=FEATURE_COLUMNS,
    target_columns=[target],
    n_components=6,
    prefix="pca6",
)

## 2. Readout helpers

In [3]:
def robust_topk_readout(result, sequence_splits, *, top_k=120, alpha=3000.0):
    H_train = result.train_features
    H_val = result.val_features
    H_test = result.test_features

    _, y_train, _ = sequence_splits["train"]
    _, y_val, _ = sequence_splits["val"]
    _, y_test, _ = sequence_splits["test"]

    lower = np.percentile(H_train, 1.0, axis=0)
    upper = np.percentile(H_train, 99.0, axis=0)
    H_train = np.clip(H_train, lower, upper)
    H_val = np.clip(H_val, lower, upper)
    H_test = np.clip(H_test, lower, upper)

    corr = _safe_feature_target_correlations(H_train, y_train)
    k = min(top_k, H_train.shape[1])
    idx = np.argsort(np.abs(corr))[-k:]
    H_train = H_train[:, idx]
    H_val = H_val[:, idx]
    H_test = H_test[:, idx]

    scaler = StandardScaler()
    H_train_s = scaler.fit_transform(H_train)
    H_val_s = scaler.transform(H_val)
    H_test_s = scaler.transform(H_test)

    model = Ridge(alpha=alpha)
    model.fit(H_train_s, np.log(np.maximum(y_train, 1e-8)))

    pred_train = np.exp(model.predict(H_train_s))
    pred_val = np.exp(model.predict(H_val_s))
    pred_test = np.exp(model.predict(H_test_s))

    return {
        "pred_train": pred_train,
        "pred_val": pred_val,
        "pred_test": pred_test,
        "H_train": H_train,
        "H_val": H_val,
        "H_test": H_test,
        "selected_idx": idx,
        "top_k_used": k,
    }


def split_metrics(y_train, y_val, y_test, pred_train, pred_val, pred_test):
    out = {}
    for name, y, pred in [
        ("train", y_train, pred_train),
        ("val", y_val, pred_val),
        ("test", y_test, pred_test),
    ]:
        m = evaluate_volatility_forecast(y, pred)
        out[f"{name}_rmse"] = m.rmse
        out[f"{name}_qlike"] = m.qlike
        out[f"{name}_mz_r2"] = m.mz_r2
        out[f"{name}_mz_beta"] = m.mz_beta
        out[f"{name}_pred_mean"] = float(np.mean(pred))
        out[f"{name}_pred_std"] = float(np.std(pred))
        out[f"{name}_corr"] = float(np.corrcoef(y, pred)[0, 1])
    return out


def high_vol_stats(y_train, y, pred, q=0.80):
    threshold = np.quantile(y_train, q)
    actual_high = y >= threshold
    pred_high = pred >= threshold
    tp = int(np.sum(actual_high & pred_high))
    fp = int(np.sum(~actual_high & pred_high))
    fn = int(np.sum(actual_high & ~pred_high))
    return {
        "threshold": float(threshold),
        "actual_high_rate": float(actual_high.mean()),
        "pred_high_rate": float(pred_high.mean()),
        "high_vol_recall": tp / max(tp + fn, 1),
        "high_vol_precision": tp / max(tp + fp, 1),
        "tp": tp,
        "fp": fp,
        "fn": fn,
    }

## 3. Temporal-resolution runs

In [4]:
temporal_configs = [
    {"run_name": "qrc_temporal_anchor6_even", "lookback_days": 40, "anchor_count": 6, "anchor_policy": "even"},
    {"run_name": "qrc_temporal_anchor10_even", "lookback_days": 40, "anchor_count": 10, "anchor_policy": "even"},
    {"run_name": "qrc_temporal_anchor20_even", "lookback_days": 40, "anchor_count": 20, "anchor_policy": "even"},
    {"run_name": "qrc_temporal_anchor10_recent", "lookback_days": 40, "anchor_count": 10, "anchor_policy": "recent"},
    {"run_name": "qrc_temporal_anchor20_recent", "lookback_days": 40, "anchor_count": 20, "anchor_policy": "recent"},
]

rows = []
diag_rows = []
hv_rows = []

for cfg in temporal_configs:
    print("Running", cfg["run_name"])

    seq = make_qrc_sequence_splits(
        pca6.splits,
        feature_columns=pca6.feature_columns,
        target_column=target,
        lookback_days=cfg["lookback_days"],
    )
    _, y_train, _ = seq["train"]
    _, y_val, _ = seq["val"]
    _, y_test, _ = seq["test"]

    qrc_config = TFIMQRCConfig(
        qubits=6,
        pca_components=6,
        lookback_days=cfg["lookback_days"],
        anchor_count=cfg["anchor_count"],
        anchor_policy=cfg["anchor_policy"],
        observable_mode="zxzz",
        collect_anchor_features=True,
        topology="full",
        trotter_steps_per_anchor=3,
        virtual_nodes_per_anchor=3,
        coupling_scale=0.7,
        transverse_field=0.5,
        evolution_time=0.5,
        angle_max=np.pi / 2,
        ridge_alpha=3000.0,
        target_transform="log",
        seed=42,
        use_disorder=True,
        disorder_strength=0.20,
    )

    base_result = fit_tfim_qrc_regressor(seq, config=qrc_config, target=target, verbose=True)
    readout = robust_topk_readout(base_result, seq, top_k=120, alpha=3000.0)

    row = {
        **cfg,
        "n_raw_features": base_result.train_features.shape[1],
        "n_selected_features": readout["top_k_used"],
    }
    row.update(split_metrics(
        y_train, y_val, y_test,
        readout["pred_train"], readout["pred_val"], readout["pred_test"],
    ))
    row.update({f"test_{k}": v for k, v in high_vol_stats(y_train, y_test, readout["pred_test"]).items()})
    rows.append(row)

    diag = diagnose_reservoir_feature_splits(
        readout["H_train"], readout["H_val"], readout["H_test"],
        y_train, y_val, y_test,
    )
    diag.insert(0, "run_name", cfg["run_name"])
    diag.insert(1, "anchor_count", cfg["anchor_count"])
    diag.insert(2, "anchor_policy", cfg["anchor_policy"])
    diag_rows.append(diag)

    for split_name, y, pred in [
        ("train", y_train, readout["pred_train"]),
        ("val", y_val, readout["pred_val"]),
        ("test", y_test, readout["pred_test"]),
    ]:
        hv = high_vol_stats(y_train, y, pred)
        hv.update({"run_name": cfg["run_name"], "split": split_name, "anchor_count": cfg["anchor_count"], "anchor_policy": cfg["anchor_policy"]})
        hv_rows.append(hv)

temporal_results = pd.DataFrame(rows)
temporal_diagnostics = pd.concat(diag_rows, ignore_index=True)
temporal_high_vol = pd.DataFrame(hv_rows)

temporal_results.sort_values(["test_rmse", "test_qlike"], ascending=[True, True])

Running qrc_temporal_anchor6_even
QRC sample 0/5420
QRC sample 250/5420
QRC sample 500/5420
QRC sample 750/5420
QRC sample 1000/5420
QRC sample 1250/5420
QRC sample 1500/5420
QRC sample 1750/5420
QRC sample 2000/5420
QRC sample 2250/5420
QRC sample 2500/5420
QRC sample 2750/5420
QRC sample 3000/5420
QRC sample 3250/5420
QRC sample 3500/5420
QRC sample 3750/5420
QRC sample 4000/5420
QRC sample 4250/5420
QRC sample 4500/5420
QRC sample 4750/5420
QRC sample 5000/5420
QRC sample 5250/5420
QRC sample 0/1219
QRC sample 250/1219
QRC sample 500/1219
QRC sample 750/1219
QRC sample 1000/1219
QRC sample 0/1019
QRC sample 250/1019
QRC sample 500/1019
QRC sample 750/1019
QRC sample 1000/1019
Running qrc_temporal_anchor10_even
QRC sample 0/5420
QRC sample 250/5420
QRC sample 500/5420
QRC sample 750/5420
QRC sample 1000/5420
QRC sample 1250/5420
QRC sample 1500/5420
QRC sample 1750/5420
QRC sample 2000/5420
QRC sample 2250/5420
QRC sample 2500/5420
QRC sample 2750/5420
QRC sample 3000/5420
QRC sample

,run_name,lookback_days,anchor_count,anchor_policy,n_raw_features,n_selected_features,train_rmse,train_qlike,train_mz_r2,train_mz_beta,...,test_pred_std,test_corr,test_threshold,test_actual_high_rate,test_pred_high_rate,test_high_vol_recall,test_high_vol_precision,test_tp,test_fp,test_fn
3,qrc_temporal_anchor10_recent,40,10,recent,459,120,0.084755,-2.479928,0.306357,1.286031,...,0.041984,0.344242,0.213021,0.23945,0.174681,0.352459,0.483146,86,92,158
0,qrc_temporal_anchor6_even,40,6,even,306,120,0.083086,-2.506907,0.342733,1.354734,...,0.040356,0.309921,0.213021,0.23945,0.150147,0.323770,0.516340,79,74,165
4,qrc_temporal_anchor20_recent,40,20,recent,816,120,0.085697,-2.467280,0.292340,1.313067,...,0.039318,0.278539,0.213021,0.23945,0.123651,0.250000,0.484127,61,65,183
1,qrc_temporal_anchor10_even,40,10,even,510,120,0.084849,-2.483433,0.307199,1.313304,...,0.039530,0.270553,0.213021,0.23945,0.133464,0.200820,0.360294,49,87,195
2,qrc_temporal_anchor20_even,40,20,even,1020,120,0.086796,-2.451557,0.269274,1.273005,...,0.038315,0.222082,0.213021,0.23945,0.128557,0.204918,0.381679,50,81,194


## 4. High-volatility recall view

In [5]:
temporal_high_vol.sort_values(["split", "high_vol_recall"], ascending=[True, False])

,threshold,actual_high_rate,pred_high_rate,high_vol_recall,high_vol_precision,tp,fp,fn,run_name,split,anchor_count,anchor_policy
11,0.213021,0.239450,0.174681,0.352459,0.483146,86,92,158,qrc_temporal_anchor10_recent,test,10,recent
2,0.213021,0.239450,0.150147,0.323770,0.516340,79,74,165,qrc_temporal_anchor6_even,test,6,even
14,0.213021,0.239450,0.123651,0.250000,0.484127,61,65,183,qrc_temporal_anchor20_recent,test,20,recent
8,0.213021,0.239450,0.128557,0.204918,0.381679,50,81,194,qrc_temporal_anchor20_even,test,20,even
5,0.213021,0.239450,0.133464,0.200820,0.360294,49,87,195,qrc_temporal_anchor10_even,test,10,even
0,0.213021,0.200000,0.093173,0.292435,0.627723,317,188,767,qrc_temporal_anchor6_even,train,6,even
9,0.213021,0.200000,0.102214,0.273063,0.534296,296,258,788,qrc_temporal_anchor10_recent,train,10,recent
3,0.213021,0.200000,0.081550,0.232472,0.570136,252,190,832,qrc_temporal_anchor10_even,train,10,even
12,0.213021,0.200000,0.069004,0.168819,0.489305,183,191,901,qrc_temporal_anchor20_recent,train,20,recent
6,0.213021,0.200000,0.062546,0.164207,0.525074,178,161,906,qrc_temporal_anchor20_even,train,20,even


## 5. Selected-feature diagnostics

In [6]:
temporal_diagnostics.sort_values(["run_name", "split"])

,run_name,anchor_count,anchor_policy,split,n_samples,n_features,near_constant_features,feature_std_min,feature_std_median,feature_std_max,effective_rank,condition_number,mean_abs_feature_target_corr,max_abs_feature_target_corr,mean_abs_shift_vs_train,max_abs_shift_vs_train
5,qrc_temporal_anchor10_even,10,even,test,1019,120,0,0.165176,0.254824,0.447752,53.218275,1051.700485,0.144378,0.283414,0.154421,0.384596
3,qrc_temporal_anchor10_even,10,even,train,5420,120,0,0.166751,0.250482,0.457084,53.909025,1303.281877,0.237175,0.469685,0.000000,0.000000
4,qrc_temporal_anchor10_even,10,even,val,1219,120,0,0.150295,0.237661,0.422006,53.133895,1870.982183,0.129489,0.245581,0.120960,0.346150
11,qrc_temporal_anchor10_recent,10,recent,test,1019,120,0,0.165603,0.259877,0.447752,52.350530,690.775659,0.211849,0.348781,0.177190,0.384596
9,qrc_temporal_anchor10_recent,10,recent,train,5420,120,0,0.164641,0.262354,0.457084,53.410910,594.490110,0.262146,0.469685,0.000000,0.000000
10,qrc_temporal_anchor10_recent,10,recent,val,1219,120,0,0.169247,0.246702,0.422006,52.125014,697.781796,0.100718,0.253451,0.095158,0.353389
8,qrc_temporal_anchor20_even,20,even,test,1019,120,0,0.140038,0.308974,0.447752,55.917661,1195.915619,0.130202,0.229636,0.149293,0.384596
6,qrc_temporal_anchor20_even,20,even,train,5420,120,0,0.146431,0.289686,0.457084,56.524731,1433.660195,0.235291,0.469685,0.000000,0.000000
7,qrc_temporal_anchor20_even,20,even,val,1219,120,0,0.145953,0.268707,0.422006,55.368435,2372.507532,0.103967,0.171922,0.156171,0.364588
14,qrc_temporal_anchor20_recent,20,recent,test,1019,120,0,0.149125,0.212518,0.447752,58.287153,897.450641,0.182535,0.291731,0.146272,0.384596


## 6. Save outputs

In [7]:
out_dir = Path("results/tables")
out_dir.mkdir(parents=True, exist_ok=True)

temporal_results.to_csv(out_dir / "phase2_qrc_temporal_resolution_ablation.csv", index=False)
temporal_diagnostics.to_csv(out_dir / "phase2_qrc_temporal_resolution_diagnostics.csv", index=False)
temporal_high_vol.to_csv(out_dir / "phase2_qrc_temporal_resolution_high_vol.csv", index=False)

print("Saved temporal-resolution ablation tables to", out_dir)

Saved temporal-resolution ablation tables to results/tables


## Reference

Current best QRC before this ablation:

```text
anchor_count = 6, anchor_policy = even
test_rmse  = 0.100530
test_qlike = -2.045391
test_mz_r2 = 0.096051
test high-vol recall ≈ 0.324
```

ESN diagnostic reference from previous notebook:

```text
test_rmse  ≈ 0.088095
test_qlike ≈ -2.455326
test_mz_r2 ≈ 0.447548
test high-vol recall ≈ 0.689
```